In [1]:
# Creating chatbot using Hugging face 
# Set up hugging face inference 
import os
from huggingface_hub import InferenceClient

client = InferenceClient()

# Set initial system message
content = """
You are OrderBot, an automated assistant for FoodHut.

Rules:
- Greet the customer warmly and wait for user's prompt
- Ask what they’d like to order.
- Do NOT list the entire menu unless the customer asks.
- Clarify sizes, toppings, and extras only when relevant.
- Collect the entire order before moving on.
- After collecting the order, ask if it is pickup or delivery.
- If delivery, ask for the address.
- Summarize the order and confirm before payment.
- Keep replies short, friendly, and conversational.

Menu:
- Pizzas:
  - Pepperoni: Large $12.95, Medium $10.00, Small $7.00
  - Cheese: Large $10.95, Medium $9.25, Small $6.50
  - Eggplant: Large $11.95, Medium $9.75, Small $6.75
- Sides:
  - Fries: Large $4.50, Small $3.50
  - Greek Salad: $7.25
- Toppings:
  - Extra cheese $2.00
  - Mushrooms $1.50
  - Sausage $3.00
  - Canadian bacon $3.50
  - AI sauce $1.50
  - Peppers $1.00
- Drinks:
  - Coke: Large $3.00, Medium $2.00, Small $1.00
  - Sprite: Large $3.00, Medium $2.00, Small $1.00
  - Bottled water: $5.00

"""

chats = [{'role': 'system', 'content': content}]  # Accumulate messages


# model="mistralai/Mistral-7B-Instruct-v0.2"

def get_completion(prompt):
    global chats
    chats.append({"role": "user", "content": prompt}) # add user messages to chats for context
    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages=chats,
        temperature=0,
    )
    reply = response.choices[0].message["content"]  # add assistant messages to chats for context
    chats.append({"role": "assistant", "content": reply})
    return reply

def reset_chats():
    global chats
    chats = [{'role': 'system', 'content': content}]


In [3]:
# Create UI

import ipywidgets as widgets
from IPython.display import display

# Chat display area
chat_area = widgets.HTML(value="", layout=widgets.Layout(width="100%", height="300px", overflow="auto", border="1px solid gray", padding="10px"))

# Input + send button
input_box = widgets.Text(placeholder="Type your message here...")
send_button = widgets.Button(description="Send", button_style="success")

# Function to handle sending
def on_send(_):
    user_msg = input_box.value.strip()
    if not user_msg:
        return
    input_box.value = ""

    # Append user message
    chat_area.value += f"<p><b>You:</b> {user_msg}</p>"

    # Get bot reply
    bot_reply = get_completion(user_msg)
    chat_area.value += f"<p><b>Bot:</b> {bot_reply}</p>"

# Bind button click
send_button.on_click(on_send)

# Layout
ui = widgets.VBox([chat_area, widgets.HBox([input_box, send_button])])

# Show initial bot greeting
initial_bot_reply = get_completion("Hello")
chat_area.value += f"<p><b>Bot:</b> {initial_bot_reply}</p>"

# Reset button
reset_button = widgets.Button(description="Reset", button_style="warning")

def on_reset(_):
    # Reset the chat history
    reset_chats()
    # Clear the chat area
    chat_area.value = ""
    # Show initial bot greeting again
    initial_bot_reply = get_completion("Hello")
    chat_area.value += f"<p><b>Bot:</b> {initial_bot_reply}</p>"

reset_button.on_click(on_reset)

# Layout with reset button included
ui = widgets.VBox([
    chat_area,
    widgets.HBox([input_box, send_button, reset_button])
])

display(ui)

# Show initial bot greeting
initial_bot_reply = get_completion("Hello")
chat_area.value += f"<p><b>Bot:</b> {initial_bot_reply}</p>"

